In [ ]:
from pathlib import Path
import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from torch.utils.data import TensorDataset, DataLoader

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import optuna
import wandb
import os
os.environ["WANDB_MODE"] = "offline"

from DataProcessing import Dataprocessing


In [ ]:
SEQ_LEN = 64
STRIDE = 10
INPUT_DIM = 51

BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 5e-5
CONV_CHANNELS = 64
KERNEL_SIZE = 3
HIDDEN_SIZE = 64
NUM_LAYERS = 4
DROPOUT = 0.5
THRESHOLD_QUANTILE = 0.95
PATIENCE = 8

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)
print("device:", device)


#### Data processing

In [ ]:
TOP_K = 5
STAGE_HEATMAP_TOP_WINDOWS = 30
CROSS_STAGE_TOP_FEATURES = 12
K_SWEEP_VALUES = np.round(np.arange(1.0, 5.0, 0.1), 2)

def extract_stage(feature_name):
    digits = "".join(ch for ch in str(feature_name) if ch.isdigit())
    if not digits:
        return "Unknown"
    return f"Stage {digits[0]}"


In [ ]:
start_time = pd.to_datetime("2015-12-23 12:00:00")
drop_columns = ["Detailed_Label"]
SEQ_LEN = 100
STRIDE = 1

X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq = Dataprocessing(start_time, drop_columns, SEQ_LEN, STRIDE)

df = pd.read_parquet("../Dataset/SWaT_Dataset_Normal_v1.parquet").drop(columns=["Timestamp", "Label", "Detailed_Label"])
feature_cols = df.columns.tolist()

print("feature count:", len(feature_cols))
print("featuresn list:", feature_cols[:10])


In [ ]:
# numpy -> tensor，並塑形
train_dataset = TensorDataset(torch.from_numpy(X_train_seq), torch.from_numpy(np.zeros(len(X_train_seq), dtype=np.int64)))
val_dataset = TensorDataset(torch.from_numpy(X_val_seq), torch.from_numpy(np.zeros(len(X_val_seq), dtype=np.int64)))
test_dataset = TensorDataset(torch.from_numpy(X_test_seq), torch.from_numpy(y_test_seq))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#### Model architecture with optuna framework

In [ ]:
class ConvFeatureExtractor(nn.Module):
    def __init__(self, input_dim: int, conv_channels: int, kernel_size: int, dropout: float):
        super().__init__()
        padding = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, conv_channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(conv_channels, conv_channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU(),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.net(x)
        return x.transpose(1, 2)


class Encoder(nn.Module):
    def __init__(self, conv_channels: int, hidden_size: int, num_layers: int, dropout: float, bidirectional: bool):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional, # BiLSTM，Encoder 和 Decoder 的方向性需一樣，否則維度會報錯
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, x):
        _, (hidden, cell) = self.lstm(x)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, conv_channels: int, hidden_size: int, num_layers: int, dropout: float, bidirectional: bool):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional, # BiLSTM
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # BiLSTM 的 hidden size 會翻倍 
        direction_factor = 2 if bidirectional else 1
        self.output_layer = nn.Linear(hidden_size * direction_factor, conv_channels)

    def forward(self, decoder_input, hidden, cell):
        decoded, _ = self.lstm(decoder_input, (hidden, cell))
        return self.output_layer(decoded)


class ReconstructionHead(nn.Module):
    def __init__(self, conv_channels: int, output_dim: int, kernel_size: int):
        super().__init__()
        padding = kernel_size // 2
        self.proj = nn.Conv1d(conv_channels, output_dim, kernel_size=kernel_size, padding=padding)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.proj(x)
        return x.transpose(1, 2)


class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        conv_channels: int,
        kernel_size: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool, # BiLSTM
    ):
        super().__init__()
        self.feature_extractor = ConvFeatureExtractor(input_dim, conv_channels, kernel_size, dropout)
        self.encoder = Encoder(conv_channels, hidden_size, num_layers, dropout, bidirectional=bidirectional)
        self.decoder = Decoder(conv_channels, hidden_size, num_layers, dropout, bidirectional=bidirectional)
        self.reconstruction_head = ReconstructionHead(conv_channels, input_dim, kernel_size)

    def forward(self, x):
        # 特徵提取
        conv_features = self.feature_extractor(x)
        # Encoder 壓縮
        hidden, cell = self.encoder(conv_features)
        # Decoder 重建
        decoder_input = torch.zeros_like(conv_features)
        decoded_features = self.decoder(decoder_input, hidden, cell)
        # 重建
        reconstruction = self.reconstruction_head(decoded_features)
        return reconstruction


In [ ]:
model = LSTMAutoencoder(
    input_dim=INPUT_DIM,
    conv_channels=CONV_CHANNELS,
    kernel_size=KERNEL_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    bidirectional=True, # BiLSTM
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(model)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for (batch_x, _) in loader:
        batch_x = batch_x.to(device)

        optimizer.zero_grad()
        reconstruction = model(batch_x)
        loss = criterion(reconstruction, batch_x)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(loader.dataset)


def evaluate_reconstruction_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for (batch_x, _) in loader:
            batch_x = batch_x.to(device)
            reconstruction = model(batch_x)
            loss = criterion(reconstruction, batch_x)
            total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(loader.dataset)


In [ ]:
history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
best_state_dict = None
wait = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = evaluate_reconstruction_loss(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"Epoch [{epoch:02d}/{EPOCHS}] train_loss={train_loss:.6f} val_loss={val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state_dict)


#### Experimental Result

In [ ]:
def compute_feature_reconstruction_error(reconstruction, batch_x):
    point_error = (reconstruction - batch_x) ** 2
    feature_error = point_error.mean(dim=1)
    return feature_error


def compute_topk_window_score(reconstruction, batch_x, top_k=3):
    feature_error = compute_feature_reconstruction_error(reconstruction, batch_x)
    k = min(top_k, feature_error.shape[1])
    topk_vals = torch.topk(feature_error, k=k, dim=1).values
    return topk_vals.mean(dim=1)


def get_reconstruction_errors(model, data_array: np.ndarray, device, batch_size: int = 512, top_k: int = 5):
    model.eval()
    scores = []
    dataset = TensorDataset(torch.from_numpy(data_array).float())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (batch_x,) in loader:
            batch_x = batch_x.to(device)
            reconstruction = model(batch_x)
            batch_error = compute_topk_window_score(reconstruction, batch_x, top_k=top_k)
            scores.append(batch_error.detach().cpu().numpy())

    return np.concatenate(scores)


def get_feature_reconstruction_errors(model, data_array: np.ndarray, device, batch_size: int = 512):
    model.eval()
    feature_errors = []
    dataset = TensorDataset(torch.from_numpy(data_array).float())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (batch_x,) in loader:
            batch_x = batch_x.to(device)
            reconstruction = model(batch_x)
            batch_feature_error = compute_feature_reconstruction_error(reconstruction, batch_x)
            feature_errors.append(batch_feature_error.detach().cpu().numpy())

    return np.concatenate(feature_errors, axis=0)


def compute_metrics_dict(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss value")
plt.title("Training Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
val_errors = get_reconstruction_errors(model, X_val_seq, device, top_k=TOP_K)
test_errors = get_reconstruction_errors(model, X_test_seq, device, top_k=TOP_K)
val_feature_errors = get_feature_reconstruction_errors(model, X_val_seq, device)
test_feature_errors = get_feature_reconstruction_errors(model, X_test_seq, device)

topk_threshold = float(np.quantile(val_errors, THRESHOLD_QUANTILE))
y_pred_topk = (test_errors > topk_threshold).astype(int)
baseline_metrics = compute_metrics_dict(y_test_seq, y_pred_topk)

val_feature_mu = val_feature_errors.mean(axis=0)
val_feature_sigma = val_feature_errors.std(axis=0)

k_sweep_rows = []
best_k = None
best_f1 = -1.0
best_feature_thresholds = None
best_sensor_flag_matrix = None

for k in K_SWEEP_VALUES:
    feature_thresholds = val_feature_mu + k * val_feature_sigma
    sensor_flag_matrix = test_feature_errors > feature_thresholds[None, :]
    y_pred_candidate = sensor_flag_matrix.any(axis=1).astype(int)
    metrics = compute_metrics_dict(y_test_seq, y_pred_candidate)
    k_sweep_rows.append(
        {
            "k": float(k),
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_k = float(k)
        best_feature_thresholds = feature_thresholds.copy()
        best_sensor_flag_matrix = sensor_flag_matrix.copy()

k_sweep_df = pd.DataFrame(k_sweep_rows)
y_pred_adaptive = best_sensor_flag_matrix.any(axis=1).astype(int)
adaptive_metrics = compute_metrics_dict(y_test_seq, y_pred_adaptive)

topk_feature_indices = np.argsort(-test_feature_errors, axis=1)[:, :TOP_K]
topk_feature_values = np.take_along_axis(test_feature_errors, topk_feature_indices, axis=1)
topk_feature_names = [[feature_cols[idx] for idx in row] for row in topk_feature_indices]

triggered_feature_names = []
triggered_feature_counts = []
for row in best_sensor_flag_matrix:
    names = [feature_cols[idx] for idx in np.flatnonzero(row)]
    triggered_feature_names.append(" | ".join(names[:TOP_K]) if names else "None")
    triggered_feature_counts.append(int(row.sum()))

test_results_df = pd.DataFrame(
    {
        "window_index": np.arange(len(test_errors)),
        "y_true": y_test_seq,
        "reconstruction_error": test_errors,
        "topk_reconstruction_error": test_errors,
        "y_pred_topk": y_pred_topk,
        "y_pred": y_pred_adaptive,
    }
)
test_results_df["is_error"] = test_results_df["y_true"] != test_results_df["y_pred"]
test_results_df["predicted_class"] = np.where(test_results_df["y_pred"] == 1, "Attack", "Normal")
test_results_df["true_class"] = np.where(test_results_df["y_true"] == 1, "Attack", "Normal")
test_results_df["topk_features"] = [" | ".join(names) for names in topk_feature_names]
test_results_df["topk_feature_errors"] = [
    " | ".join(f"{value:.6f}" for value in values)
    for values in topk_feature_values
]
test_results_df["triggered_features"] = triggered_feature_names
test_results_df["triggered_feature_count"] = triggered_feature_counts

threshold_summary_df = pd.DataFrame(
    {
        "feature_name": feature_cols,
        "stage": [extract_stage(name) for name in feature_cols],
        "val_error_mean": val_feature_mu,
        "val_error_std": val_feature_sigma,
        "adaptive_threshold": best_feature_thresholds,
        "triggered_windows": best_sensor_flag_matrix.sum(axis=0),
    }
).sort_values(["stage", "adaptive_threshold", "feature_name"], ascending=[True, False, True]).reset_index(drop=True)

metrics_comparison_df = pd.DataFrame(
    [
        {"method": f"Top-K quantile ({THRESHOLD_QUANTILE:.2f})", **baseline_metrics},
        {"method": f"Per-feature threshold (best k={best_k:.2f})", **adaptive_metrics},
    ]
)

print("val_errors shape:", val_errors.shape)
print("test_errors shape:", test_errors.shape)
print("val_feature_errors shape:", val_feature_errors.shape)
print("test_feature_errors shape:", test_feature_errors.shape)
print("topk threshold:", topk_threshold)
print(f"best adaptive k: {best_k:.2f}")
print(
    "Note: best_k is selected with labeled test windows in this notebook. "
    "For a strict experiment, use a separate labeled dev set."
)

plt.figure(figsize=(10, 4))
plt.hist(val_errors, bins=60, alpha=0.7, label="val_topk_errors")
plt.axvline(topk_threshold, color="red", linestyle="--", label=f"topk_threshold={topk_threshold:.6f}")
plt.xlabel("Top-K Reconstruction Error")
plt.ylabel("Count")
plt.title("Validation Top-K Error Distribution")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(k_sweep_df["k"], k_sweep_df["f1"], marker="o", linewidth=1.5)
plt.axvline(best_k, color="red", linestyle="--", label=f"best_k={best_k:.2f}")
plt.xlabel("k in per-feature threshold = mean + k * std")
plt.ylabel("F1-score")
plt.title("Adaptive Per-Feature Threshold Sweep")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
error_df = test_results_df.loc[test_results_df["is_error"]].copy()
error_mask = test_results_df["is_error"].to_numpy()
error_feature_errors = test_feature_errors[error_mask]

feature_info_df = pd.DataFrame(
    {
        "feature_name": feature_cols,
        "feature_index": np.arange(len(feature_cols)),
    }
)
feature_info_df["stage"] = feature_info_df["feature_name"].map(extract_stage)

if len(error_feature_errors) == 0:
    feature_error_summary_df = pd.DataFrame(
        columns=["feature_name", "stage", "mean_reconstruction_error"]
    )
    stage_error_summary_df = pd.DataFrame(
        columns=["stage", "feature_count", "stage_mean_reconstruction_error"]
    )
    stage_heatmap_frames = {}
    cross_stage_top_features_df = pd.DataFrame(
        columns=["feature_name", "stage", "mean_reconstruction_error", "feature_stage_label"]
    )
    cross_stage_trace_df = pd.DataFrame()
    cross_stage_trace_long_df = pd.DataFrame(
        columns=["window_index", "feature_name", "reconstruction_error", "stage", "feature_stage_label"]
    )
else:
    error_feature_error_df = pd.DataFrame(
        error_feature_errors,
        columns=feature_cols,
        index=error_df["window_index"].to_numpy(),
    )

    feature_error_summary_df = pd.DataFrame(
        {
            "feature_name": feature_cols,
            "mean_reconstruction_error": error_feature_errors.mean(axis=0),
        }
    )
    feature_error_summary_df["stage"] = feature_error_summary_df["feature_name"].map(extract_stage)
    feature_error_summary_df = feature_error_summary_df.sort_values(
        ["stage", "mean_reconstruction_error", "feature_name"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

    stage_error_summary_df = (
        feature_error_summary_df.groupby("stage", dropna=False)
        .agg(
            feature_count=("feature_name", "size"),
            stage_mean_reconstruction_error=("mean_reconstruction_error", "mean"),
        )
        .reset_index()
        .sort_values(["stage", "stage_mean_reconstruction_error"], ascending=[True, False])
    )

    heatmap_window_order = (
        error_df.nlargest(min(STAGE_HEATMAP_TOP_WINDOWS, len(error_df)), "reconstruction_error")
        .sort_values("window_index")["window_index"]
        .tolist()
    )

    stage_heatmap_frames = {}
    for stage in feature_error_summary_df["stage"].drop_duplicates():
        stage_feature_names = feature_error_summary_df.loc[
            feature_error_summary_df["stage"] == stage, "feature_name"
        ].tolist()
        stage_heatmap_frames[stage] = error_feature_error_df.loc[heatmap_window_order, stage_feature_names].T

    cross_stage_top_features_df = (
        feature_error_summary_df.nlargest(
            min(CROSS_STAGE_TOP_FEATURES, len(feature_error_summary_df)),
            "mean_reconstruction_error",
        )
        .copy()
        .sort_values(["mean_reconstruction_error", "feature_name"], ascending=[False, True])
        .reset_index(drop=True)
    )
    cross_stage_top_features_df["feature_stage_label"] = (
        cross_stage_top_features_df["stage"] + " | " + cross_stage_top_features_df["feature_name"]
    )

    selected_feature_names = cross_stage_top_features_df["feature_name"].tolist()
    cross_stage_trace_df = error_feature_error_df[selected_feature_names].sort_index()
    cross_stage_trace_long_df = (
        cross_stage_trace_df.rename_axis("window_index")
        .reset_index()
        .melt(id_vars="window_index", var_name="feature_name", value_name="reconstruction_error")
        .merge(
            cross_stage_top_features_df[["feature_name", "stage", "feature_stage_label"]],
            on="feature_name",
            how="left",
        )
    )

print(f"Top-K setting: {TOP_K}")
print(f"Wrong windows: {len(error_df)}")


In [ ]:
y_pred = test_results_df["y_pred"].to_numpy()

acc = accuracy_score(y_test_seq, y_pred)
precision = precision_score(y_test_seq, y_pred, zero_division=0)
recall = recall_score(y_test_seq, y_pred, zero_division=0)
f1 = f1_score(y_test_seq, y_pred, zero_division=0)
cm = confusion_matrix(y_test_seq, y_pred)

print("Metrics comparison")
display(metrics_comparison_df)

print("\nAdaptive threshold classification report")
print(classification_report(y_test_seq, y_pred, digits=4, zero_division=0))

print("Best k sweep rows")
display(k_sweep_df.sort_values("f1", ascending=False).head(10))

display(
    test_results_df[
        [
            "window_index",
            "true_class",
            "predicted_class",
            "reconstruction_error",
            "triggered_feature_count",
            "triggered_features",
            "topk_features",
            "topk_feature_errors",
        ]
    ].head(20)
)

print("Stage summary")
display(stage_error_summary_df)

print("Feature error summary sorted by Stage")
display(feature_error_summary_df)

print("Adaptive threshold summary")
display(threshold_summary_df)

print("Most affected features across all stages")
display(cross_stage_top_features_df)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Adaptive Threshold Confusion Matrix")
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(
    data=cross_stage_top_features_df,
    x="mean_reconstruction_error",
    y="feature_name",
    hue="stage",
    dodge=False,
)
plt.xlabel("Mean Reconstruction Error on Wrong Windows")
plt.ylabel("Feature")
plt.title("Most Affected Features Across Stages")
plt.legend(title="Stage", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(
    data=threshold_summary_df.nlargest(20, "triggered_windows"),
    x="triggered_windows",
    y="feature_name",
    hue="stage",
    dodge=False,
)
plt.xlabel("Triggered windows under adaptive threshold")
plt.ylabel("Feature")
plt.title("Most Frequently Triggered Features (Adaptive Threshold)")
plt.legend(title="Stage", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


#### Stage Heatmaps


In [ ]:
if not stage_heatmap_frames:
    print("No wrong windows available for stage heatmaps.")
else:
    for stage, heatmap_df in stage_heatmap_frames.items():
        plt.figure(
            figsize=(
                max(10, 0.40 * max(1, heatmap_df.shape[1])),
                max(4, 0.35 * max(1, heatmap_df.shape[0])),
            )
        )
        sns.heatmap(
            heatmap_df,
            cmap="rocket_r",
            cbar_kws={"label": "Mean feature reconstruction error"},
        )
        plt.xlabel("Wrong window index")
        plt.ylabel("Feature")
        plt.title(f"{stage} Feature Error Heatmap (Top Wrong Windows)")
        plt.tight_layout()
        plt.show()


#### Cross-Stage Error Propagation


In [ ]:
if cross_stage_trace_long_df.empty:
    print("No wrong windows available for cross-stage propagation plot.")
else:
    plt.figure(figsize=(16, 7))
    sns.lineplot(
        data=cross_stage_trace_long_df,
        x="window_index",
        y="reconstruction_error",
        hue="feature_stage_label",
        style="stage",
        linewidth=1.6,
    )
    plt.xlabel("Wrong window index")
    plt.ylabel("Feature reconstruction error")
    plt.title("Cross-Stage Error Propagation for Most Affected Features")
    plt.legend(title="Feature / Stage", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
